<h3>Set up environment</h3>

In [ ]:
!pip install transformers datasets optimum pandas pyarrow scikit-learn

<h3>Inspect Data</h3>

In [ ]:
import pandas as pd
df = pd.read_parquet("train.parquet")

# Quick peek
print("Shape:", df.shape)
df.head()

Shape: (3822, 6)


,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [ ]:
import ast
import numpy as np
import pandas as pd

def parse_techniques(row):
    # 1) If the row is already an array, list, or tuple, convert it to a Python list
    if isinstance(row, (np.ndarray, list, tuple)):
        return list(row)
    # 2) Now row should be a scalar (like a string). Check if it's None or NaN.
    if row is None or pd.isna(row):
        return []
    # 3) If it's a string, clean and attempt to parse it.
    if isinstance(row, str):
        row_str = row.strip()
        if row_str == "None":
            return []
        try:
            parsed = ast.literal_eval(row_str)
            if isinstance(parsed, list):
                return parsed
            else:
                return []
        except Exception as e:
            print(f"Error parsing row: {row_str}. Error: {e}")
            return []
    # 4) Fallback: return an empty list if nothing matches.
    return []

# Now apply the function to the 'techniques' column
df["techniques_list"] = df["techniques"].apply(parse_techniques)

In [ ]:
def normalize_technique_name(t):
    # e.g. convert " loaded_language " -> "loaded_language"
    return t.strip().lower().replace(" ", "_")

df["techniques_list"] = df["techniques_list"].apply(
    lambda lst: [normalize_technique_name(x) for x in lst]
)


In [ ]:
df[["techniques", "techniques_list"]].head(10)


,techniques,techniques_list
0,"[euphoria, loaded_language]","[euphoria, loaded_language]"
1,"[loaded_language, cherry_picking]","[loaded_language, cherry_picking]"
2,"[loaded_language, euphoria]","[loaded_language, euphoria]"
3,None,[]
4,[loaded_language],[loaded_language]
5,[loaded_language],[loaded_language]
6,None,[]
7,"[loaded_language, glittering_generalities, eup...","[loaded_language, glittering_generalities, eup..."
8,"[cherry_picking, cliche]","[cherry_picking, cliche]"
9,"[loaded_language, cherry_picking, appeal_to_fear]","[loaded_language, cherry_picking, appeal_to_fear]"


In [ ]:
row_index = 4  # or any row that looked suspicious
print("Original:", df.loc[row_index, "techniques"])
print("Parsed:  ", df.loc[row_index, "techniques_list"])


Original: ['loaded_language']
Parsed:   ['loaded_language']


In [ ]:
all_techniques = set()
for row in df["techniques_list"]:
    for t in row:
        all_techniques.add(t)

# Sort them so we have a stable order
all_techniques_list = sorted(list(all_techniques))
tech2id = {tech: i for i, tech in enumerate(all_techniques_list)}

num_labels = len(all_techniques_list)
print("Unique techniques:", all_techniques_list)
print("Number of unique techniques:", num_labels)

Unique techniques: ['appeal_to_fear', 'bandwagon', 'cherry_picking', 'cliche', 'euphoria', 'fud', 'glittering_generalities', 'loaded_language', 'straw_man', 'whataboutism']
Number of unique techniques: 10


<h3>Create Multilabel Vectors</3>

In [ ]:
import numpy as np

def build_label_vector(tech_list):
    vec = np.zeros(num_labels, dtype=float)
    for t in tech_list:
        # Check if t is in the dictionary
        if t in tech2id:
            idx = tech2id[t]
            vec[idx] = 1.0
        else:
            print("Unrecognized technique:", repr(t))
    return vec

df["label_vector"] = df["techniques_list"].apply(build_label_vector)

In [ ]:
row_index = 4
print("Techniques list:", df.loc[row_index, "techniques_list"])
print("Label vector:", df.loc[row_index, "label_vector"])

Techniques list: ['loaded_language']
Label vector: [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]


In [ ]:
df.rename(columns={"content": "text"}, inplace=True)

use_cols = ["text", "label_vector"]
subset_df = df[use_cols].copy()
subset_df.head()

,text,label_vector
0,Новий огляд мапи DeepState від російського вій...,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ..."
1,Недавно 95 квартал жёстко поглумился над русск...,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
2,🤩\nТим часом йде евакуація Бєлгородського авто...,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, ..."
3,В Україні найближчим часом мають намір посилит...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(subset_df)
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

train_dataset, eval_dataset

(Dataset({
     features: ['text', 'label_vector'],
     num_rows: 3057
 }),
 Dataset({
     features: ['text', 'label_vector'],
     num_rows: 765
 }))

<h3>Load LiBERTa and Tokenizer</h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "Goader/liberta-large-v2"
num_labels = 10  # e.g., if you have 10 unique techniques

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

The repository for Goader/liberta-large-v2 contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Goader/liberta-large-v2.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Goader/liberta-large-v2 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<h3>Tokenize Dataset</h3>

In [ ]:
def tokenize_and_align_labels(example):
    # Tokenize text
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    # Convert label_vector (numpy array) to a list or Python scalar
    # so it can be stored in the dataset's "labels" field
    tokenized["labels"] = list(example["label_vector"])
    return tokenized

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=False)
eval_dataset  = eval_dataset.map(tokenize_and_align_labels, batched=False)

Map:   0%|          | 0/3057 [00:00<?, ? examples/s]

Map:   0%|          | 0/765 [00:00<?, ? examples/s]

<h3>Mapped Dataset to PyTorch</h3>

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

<h3>Compute Metrics</h3>

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits -> probabilities using sigmoid
    probs = 1.0 / (1.0 + np.exp(-logits))
    # Threshold at 0.5
    preds = (probs > 0.5).astype(int)
    # Compute macro-F1
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"macro_f1": f1}

<h3>Training</h3>

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results-liberta",
    evaluation_strategy="steps",
    eval_steps=100,             # Evaluate every 100 steps
    save_steps=100,
    num_train_epochs=2,         # Adjust as needed
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,                  # Mixed precision
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    # The important part:
    report_to="none"  # disables wandb & other loggers
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Macro F1
100,0.330000,0.311510,0.016981
200,0.342400,0.295137,0.128523
300,0.280700,0.271639,0.133230
400,0.290000,0.264970,0.149273
500,0.262700,0.262628,0.155366
600,0.265100,0.260928,0.184246
700,0.276800,0.254160,0.208312
800,0.243900,0.254429,0.243179
900,0.234500,0.257463,0.210250
1000,0.270800,0.254648,0.272708


TrainOutput(global_step=1530, training_loss=0.2651224313997755, metrics={'train_runtime': 1475.7944, 'train_samples_per_second': 4.143, 'train_steps_per_second': 1.037, 'total_flos': 1424495592373248.0, 'train_loss': 0.2651224313997755, 'epoch': 2.0})

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

Evaluation results: {'eval_loss': 0.2546479403972626, 'eval_macro_f1': 0.272707836134494, 'eval_runtime': 5.9035, 'eval_samples_per_second': 129.585, 'eval_steps_per_second': 32.523, 'epoch': 2.0}


In [ ]:
trainer.save_model("./liberta_finetuned")
tokenizer.save_pretrained("./liberta_finetuned")

('./liberta_finetuned/tokenizer_config.json',
 './liberta_finetuned/special_tokens_map.json',
 './liberta_finetuned/spm.model',
 './liberta_finetuned/added_tokens.json')

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./liberta_finetuned"
reloaded_tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(model_path, trust_remote_code=True)

The repository for ./liberta_finetuned contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/./liberta_finetuned.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


<h3>Inference</h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

model_path = "./liberta_finetuned"  # or the path where you saved your model

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, trust_remote_code=True)
model.eval()  # put in eval mode

# Example text to classify
sample_text = "Пакують українців у мінівени!"

# Tokenize
inputs = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Convert logits to probabilities (sigmoid for multi-label)
probs = torch.sigmoid(logits).squeeze().cpu().numpy()

# For each label dimension, threshold at 0.5
preds = (probs > 0.05).astype(int)

print("Probabilities:", probs)
print("Binary predictions:", preds)


Probabilities: [0.02624063 0.01794931 0.03295594 0.03379123 0.02484297 0.01864652
 0.02175771 0.09392028 0.01042274 0.01334797]
Binary predictions: [0 0 0 0 0 0 0 1 0 0]


<h3>Mapping back to Technique Names</h3>

In [ ]:
# Suppose tech2id = {"loaded_language": 0, "cherry_picking": 1, ...}
id2tech = {v: k for k, v in tech2id.items()}

def decode_predictions(pred_vector):
    # pred_vector might be [1,0,1,0,...]
    # Return a list of technique names
    return [id2tech[i] for i, val in enumerate(pred_vector) if val == 1]

# Example usage
tech_labels = decode_predictions(preds)
print("Predicted techniques:", tech_labels)

Predicted techniques: ['loaded_language']


<h3>Model Upload</h3>

In [ ]:
!pip install huggingface_hub

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Then in your Python code:
model.push_to_hub("taleef/liberta_finetuned", use_temp_dir=False)
tokenizer.push_to_hub("taleef/liberta_finetuned", use_temp_dir=False)

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/taleef/liberta_finetuned/commit/03df798f1913ea574ca8244d05b1b2a413fa2ca5', commit_message='Upload tokenizer', commit_description='', oid='03df798f1913ea574ca8244d05b1b2a413fa2ca5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/taleef/liberta_finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='taleef/liberta_finetuned'), pr_revision=None, pr_num=None)